### Week 4: **Capstone — Predicting Internship Selection Outcomes**
**Goal:** Use applicant information to explore which factors are associated with being selected for the NextGenLearners internship, and communicate the findings honestly to a non-technical audience.

### Step 1: Set Up
Importing the libraries used throughout this notebook: 
- pandas and numpy for data.
- matplotlib for charts.
- scikit-learn for the machine learning model and its evaluation.

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay

Exception ignored in PyObject_HasAttr(); consider using PyObject_HasAttrWithError(), PyObject_GetOptionalAttr() or PyObject_GetAttr():
Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 491, in _call_with_frames_removed
ImportError: PyCapsule_Import could not import module "pandas"


ImportError: DLL load failed while importing testing: An Application Control policy has blocked this file.

### Step 2: Load the Cleaned Dataset
We reuse the cleaned applicant data from Week 1 (duplicates removed, text standardized, missing values handled). `Phone` is read as text because phone numbers are identifiers, not quantities.

In [ ]:
df = pd.read_csv('applicants_cleaned.csv', dtype={"Phone": str})
print(df.shape)

(71, 7)


In [ ]:
print(df["Status"].value_counts())
df.head()


Status
Under Review    34
Selected        23
Rejected        14
Name: count, dtype: int64


,Applicant Name,Email,Phone,Domain Applied,University,Application Date,Status
0,Hassan Malik,hassan.malik19@gmail.com,03235456272,Data Science,FAST NUCES,27-05-2026,Under Review
1,Danish Khan,danish.khan44@gmail.com,03352156902,Mobile App Development,COMSATS University,16-08-2026,Under Review
2,Omar Sheikh,omar.sheikh47@gmail.com,03281170307,Data Science,University of Karachi,20-08-2026,Under Review
3,Bilal Iqbal,bilal.iqbal2@gmail.com,03255437923,Graphic Design,IBA Karachi,22-02-2026,Rejected
4,Talha Qureshi,talha.qureshi72@gmail.com,03241368085,Graphic Design,IBA Karachi,22-08-2026,Rejected


### Step 2b: Simulating New Features

⚠️ **Honesty note:** The four features below are **simulated**, not real historical data. They were generated by code, and three of them were deliberately set up to be somewhat higher for selected applicants. Any pattern the model finds therefore reflects the assumptions built into this simulation, **not** real behaviour at NextGenLearners. In a real deployment, this information would have to be collected from actual applications (a skills field, a portfolio URL field, etc.) before the model could be trusted.

| Feature | Type | How it was simulated |
|---|---|---|
| num_skills_listed | Number (1–8) | Pure random, unrelated to selection |
| has_portfolio | Yes/No | ~70% Yes for selected, ~35% for others |
| prior_hackathon_participation | Yes/No | ~45% Yes for selected, ~25% for others |
| statement_quality_score | Number (1–10) | Centered around 7 for selected, 5 for others |

`np.random.seed(42)` makes the "random" numbers identical on every run, so results are reproducible.

In [ ]:
np.random.seed(42)
n = len(df)
is_selected = (df["Status"] == "Selected")

df["num_skills_listed"] = np.random.randint(1, 9, size=n)

df["has_portfolio"] = np.where(
    is_selected,
    np.random.choice(["Yes", "No"], size=n, p=[0.70, 0.30]),
    np.random.choice(["Yes", "No"], size=n, p=[0.35, 0.65]),
)

df["prior_hackathon_participation"] = np.where(
    is_selected,
    np.random.choice(["Yes", "No"], size=n, p=[0.45, 0.55]),
    np.random.choice(["Yes", "No"], size=n, p=[0.25, 0.75]),
)

scores = np.random.normal(loc=np.where(is_selected, 7, 5), scale=1.5, size=n)
df["statement_quality_score"] = np.clip(scores.round(), 1, 10).astype(int)

df.head()

,Applicant Name,Email,Phone,Domain Applied,University,Application Date,Status,num_skills_listed,has_portfolio,prior_hackathon_participation,statement_quality_score
0,Hassan Malik,hassan.malik19@gmail.com,03235456272,Data Science,FAST NUCES,27-05-2026,Under Review,7,No,No,5
1,Danish Khan,danish.khan44@gmail.com,03352156902,Mobile App Development,COMSATS University,16-08-2026,Under Review,4,Yes,Yes,5
2,Omar Sheikh,omar.sheikh47@gmail.com,03281170307,Data Science,University of Karachi,20-08-2026,Under Review,5,No,No,5
3,Bilal Iqbal,bilal.iqbal2@gmail.com,03255437923,Graphic Design,IBA Karachi,22-02-2026,Rejected,7,No,Yes,8
4,Talha Qureshi,talha.qureshi72@gmail.com,03241368085,Graphic Design,IBA Karachi,22-08-2026,Rejected,3,No,No,4


**What this cell did:** 

Added four simulated columns to every applicant. 

Three (`has_portfolio`, `prior_hackathon_participation`, `statement_quality_score`) were made slightly more favourable for selected applicants. `num_skills_listed` is pure random noise with no link to selection.

In [ ]:
print(df.shape)
print(df.isnull().sum())
df.groupby("Status")[["num_skills_listed", "statement_quality_score"]].mean().round(2)

(71, 11)
Applicant Name                   0
Email                            0
Phone                            0
Domain Applied                   0
University                       0
Application Date                 2
Status                           0
num_skills_listed                0
has_portfolio                    0
prior_hackathon_participation    0
statement_quality_score          0
dtype: int64


,num_skills_listed,statement_quality_score
Status,,
Rejected,5.07,4.64
Selected,4.78,7.39
Under Review,4.47,5.41


**Check results:** 

The dataset now has 71 rows and 11 columns. The only missing values are 2 application dates carried over from Week 1. They won't be used as a feature, so they don't matter here. Selected applicants average a statement score of about 7.4, against about 4.6 for rejected ones, as designed. Average skills listed is about the same across groups (roughly 4.5–5), which is expected because that feature is random.

### Day 2: **Feature preparation**

**Goal:** The goal is to turn everything into numbers, because the model can’t read “Yes” or “Web Development”.

### Step 3: Feature Preparation
Machine learning models only work with numbers. This step:
1. Keeps only applicants with a final decision (removes "Under Review")
2. Converts Yes/No columns to 1/0
3. Creates the target column (what we want to predict)
4. One-hot encodes the Domain column
5. Selects the final input columns

### A: filter, convert Yes/No, create target

In [ ]:
df_model = df[df["Status"] != "Under Review"].copy()

df_model["has_portfolio"] = df_model["has_portfolio"].map({"Yes" : 1, "No" : 0})
df_model["prior_hackathon_participation"] = df_model["prior_hackathon_participation"].map({"Yes" : 1, "No" : 0})

df_model["target"] = (df_model["Status"] == "Selected").astype(int)

print(df_model.shape)
print(df_model["target"].value_counts())
df_model[["Status", "has_portfolio", "prior_hackathon_participation", "target"]].head()



(37, 12)
target
1    23
0    14
Name: count, dtype: int64


,Status,has_portfolio,prior_hackathon_participation,target
3,Rejected,0,1,0
4,Rejected,0,0,0
6,Selected,0,0,1
8,Rejected,1,0,0
9,Rejected,0,0,0


**What this did:** Applicants still "Under Review" were removed because they have no final outcome to learn from. Yes/No columns became 1/0, and a `target` column was created (1 = Selected, 0 = Rejected). The remaining dataset has **37 rows: 23 Selected and 14 Rejected.**

### B: one-hot encoding

In [ ]:
df_encoded = pd.get_dummies(df_model, columns=["Domain Applied"], drop_first=True, dtype=int)       
# pd.get_dummies replaces the Domain column with one 0/1 column per domain.

df_encoded.head()

,Applicant Name,Email,Phone,University,Application Date,Status,num_skills_listed,has_portfolio,prior_hackathon_participation,statement_quality_score,target,Domain Applied_Digital Marketing,Domain Applied_Graphic Design,Domain Applied_Mobile App Development,Domain Applied_Web Development
3,Bilal Iqbal,bilal.iqbal2@gmail.com,03255437923,IBA Karachi,22-02-2026,Rejected,7,0,1,8,0,0,1,0,0
4,Talha Qureshi,talha.qureshi72@gmail.com,03241368085,IBA Karachi,22-08-2026,Rejected,3,0,0,4,0,0,1,0,0
6,Hira Malik,hira.malik41@gmail.com,03005148725,NED University,NaN,Selected,5,0,0,5,1,0,0,0,1
8,Hassan Ansari,hassan.ansari50@gmail.com,03495747930,NED University,22-03-2026,Rejected,7,1,0,4,0,0,0,0,0
9,Ahmed Hussain,ahmed.hussain21@gmail.com,03481131249,FAST NUCES,21-03-2026,Rejected,2,0,0,4,0,1,0,0,0


**Why one-hot encoding:** Domains have no natural order, so numbering them 1, 2, 3 would wrongly imply one domain is "more" than another. Each domain gets its own 0/1 column instead. `drop_first=True` automatically drops the first domain alphabetically (Data Science), which becomes the default: an applicant with 0 in every domain column is a Data Science applicant. This avoids redundant information and is standard practice.

**Output:** `Domain Applied` was replaced by 4 columns (Digital Marketing, Graphic Design, Mobile App Development, Web Development). `df_encoded` has 15 columns.

### C: final feature set

In [ ]:
feature_columns = [
    "num_skills_listed",
    "has_portfolio",
    "prior_hackathon_participation",
    "statement_quality_score",
]+[col for col in df_encoded.columns if col.startswith("Domain Applied_")]

X = df_encoded[feature_columns]
y = df_encoded["target"]

print(X.shape)
print(y.shape)
X.head()

(37, 8)
(37,)


,num_skills_listed,has_portfolio,prior_hackathon_participation,statement_quality_score,Domain Applied_Digital Marketing,Domain Applied_Graphic Design,Domain Applied_Mobile App Development,Domain Applied_Web Development
3,7,0,1,8,0,1,0,0
4,3,0,0,4,0,1,0,0
6,5,0,0,5,0,0,0,1
8,7,1,0,4,0,0,0,0
9,2,0,0,4,1,0,0,0


**Output:** `X` has 37 rows and 8 columns (4 applicant features + 4 domain columns), and `y` holds the 37 outcomes (1 = Selected, 0 = Rejected). The safety check confirms there are no missing values and every column is numeric.

In [ ]:
print("Missing values in X:", X.isnull().sum().sum())
print(X.dtypes)

Missing values in X: 0
num_skills_listed                        int32
has_portfolio                            int64
prior_hackathon_participation            int64
statement_quality_score                  int64
Domain Applied_Digital Marketing         int64
Domain Applied_Graphic Design            int64
Domain Applied_Mobile App Development    int64
Domain Applied_Web Development           int64
dtype: object


**Features used:** the 4 simulated applicant features plus the domain columns.

**Features deliberately excluded:**
- *Applicant Name, Email, Phone*: identifiers with no real predictive meaning, and inappropriate to feed into a selection model.
- *Status*: this is the answer itself. Including it would let the model "cheat" (data leakage).
- *University*: excluded to avoid the model learning institutional favouritism, which is a bias risk discussed in the limitations.
- *Application Date*: administrative, not a meaningful signal, and has 2 missing values.

### Day 3: **Train/test split and training the model**

### Step 4: Train/Test Split
The data is split into a **training set** (80%), which the model learns from, and a **test set** (20%), which the model never sees during training. Testing on unseen applicants shows whether the model learned real patterns or just memorized the training data, like grading a student on questions they haven't studied before.

`random_state=42` makes the split reproducible. `stratify=y` keeps the share of Selected applicants about the same in both sets. With so few rows, this stops a lucky or unlucky split from putting almost all one outcome in the test set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training Rows:", len(X_train))
print("Test Rows:", len(X_test))
print("\n Training Outcomes:\n", y_train.value_counts())
print("\n Test Outcomes:\n", y_test.value_counts())

NameError: name 'train_test_split' is not defined